# 02 - Baselines B0 y B1

B0 predice la proporción de estados ganados en `train`. B1 calcula una puntuación fija de recursos y aprende en `train` cómo convertirla en probabilidad. Ambos se ajustan por separado en A, B y C y se evalúan en `val`. Cada partida tiene el mismo peso total. Este cuaderno no usa `calib` ni `test`.

La primera celda prepara este runtime: clona el código y descarga las tablas procesadas si faltan.


In [ ]:
# Prepara el runtime: Colab puede iniciar una máquina nueva y vacía.
from pathlib import Path  # Permite manejar rutas como objetos.
import gc, sys  # gc libera memoria; sys permite importar el código del repo.
REPO_DIR = Path('/content/repo')  # Carpeta temporal donde estará el repositorio.
if not REPO_DIR.exists():  # Si no existe, este runtime aún no tiene el código.
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo  # Descarga la rama de trabajo.
else:  # Si existe, actualiza los archivos con la versión más reciente.
    !git -C /content/repo pull
%cd /content/repo  # Ejecuta los próximos comandos desde la carpeta del proyecto.
!pip install -q -r requirements.txt  # Instala las versiones de librerías requeridas.
if '/content/repo/src' not in sys.path:  # Evita agregar la misma ruta dos veces.
    sys.path.insert(0, '/content/repo/src')  # Hace importable el paquete del proyecto.
# Quita módulos viejos de memoria para que Python use el código recién descargado.
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
DATA_DIR = Path('/content/datos')  # Aquí se guardan temporalmente las tablas Parquet.
from ptcg_wp.ingest import load_days  # Lee la lista de días incluida en la configuración.
dias = load_days('configs/dias.yaml')  # Obtiene los días cuyos datos espera el proyecto.
# Lista los archivos de estados y el archivo que asigna partidas a particiones.
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(ruta.exists() for ruta in esperados):  # Si falta algún archivo, lo baja del Kaggle privado.
    from ptcg_wp.colab import setup_kaggle  # Función que obtiene el token de Kaggle.
    setup_kaggle()  # Lee KAGGLE_API_TOKEN de los Secrets o lo solicita sin mostrarlo.
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos  # Baja tablas ya procesadas; no vuelve a procesar partidas.
assert all(ruta.exists() for ruta in esperados), 'Faltan tablas: revisa la descarga anterior.'  # Detiene el cuaderno si falta algún archivo.
# Importa herramientas usadas en las celdas siguientes.
import numpy as np  # Arreglos y operaciones numéricas.
import pandas as pd  # Tablas de datos.
from IPython.display import display  # Muestra tablas con formato en Colab.
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss  # Métricas de clasificación.
from ptcg_wp.baselines import (ScenarioStream, b0_and_hp_unit,  # Lectura por día y cálculo de B0/escala de HP.
                               b1_probability, fit_b1_mapping)  # Ajuste y predicción de B1.
SCENARIOS = ('A', 'B', 'C')  # Evalúa los tres diseños de partición acordados.
print('Entorno listo; tablas disponibles:', len(list(DATA_DIR.glob('*.parquet'))), '| escenarios:', SCENARIOS)  # Confirma la preparación.


## Peso igual por partida

Una partida con 200 estados da peso `1/200` a cada estado; una con 20 da `1/20`. Cada partida suma 1 en total. Para una métrica por etapa, los pesos se recalculan dentro de esa etapa.

In [ ]:
# Da a cada partida el mismo peso total, aunque aporte distinta cantidad de estados.
def pesos_por_partida(game_ids):
    ids = pd.Series(game_ids, copy=False)  # Vista tipo columna para usar operaciones de pandas.
    conteos = ids.value_counts(sort=False)  # Cuenta cuántos estados aporta cada partida.
    return ids.map(conteos).rdiv(1.0).to_numpy(dtype=float)  # Cada estado recibe 1/conteo de su partida.

# Compara resultados reales con probabilidades predichas y calcula tres métricas.
def metricas(y, p, game_ids):
    w = pesos_por_partida(game_ids)  # Pesos para que cada partida contribuya por igual.
    y = np.asarray(y, dtype=int)  # Convierte resultados reales a números 0 y 1.
    p = np.clip(np.asarray(p, dtype=float), 1e-15, 1 - 1e-15)  # Evita probabilidades exactamente 0 o 1.
    return {  # Devuelve métricas y tamaños de la muestra evaluada.
        'auc': roc_auc_score(y, p, sample_weight=w),  # Mide qué tan bien ordena estados positivos sobre negativos.
        'log_loss': log_loss(y, p, sample_weight=w, labels=[0, 1]),  # Penaliza errores de probabilidad; menor es mejor.
        'brier': brier_score_loss(y, p, sample_weight=w),  # Error cuadrático de probabilidad; menor es mejor.
        'n_estados': len(y),  # Número de estados incluidos.
        'n_partidas': pd.Series(game_ids).nunique(),  # Número de partidas distintas incluidas.
    }

# Crea reportes globales y separados por etapa de la partida.
def evaluar(val, pred, escenario, modelo):
    filas = []  # Aquí se acumula un diccionario de resultados por grupo.
    grupos = [('global', np.ones(len(val), dtype=bool))]  # Máscara que incluye todas las filas de val.
    grupos += [(etapa, val['stage'].to_numpy() == etapa) for etapa in ('inicio', 'medio', 'final')]  # Máscaras por etapa.
    for etapa, mascara in grupos:  # Repite los cálculos para cada grupo.
        parte = val.loc[mascara]  # Extrae únicamente las filas marcadas como True.
        if len(parte):  # Omite etapas vacías.
            # ** incorpora al reporte las métricas devueltas como diccionario.
            filas.append({'escenario': escenario, 'modelo': modelo, 'etapa': etapa, **metricas(parte.y, pred[mascara], parte.game_id)})
    return filas  # Entrega reportes para formar la tabla final.
print('Funciones de ponderación y métricas listas.')


## B0 - probabilidad a priori

B0 ignora el tablero. Calcula en train la fracción de partidas ganadas por quien actúa y predice esa probabilidad en todos los estados de val. Es el piso de comparación.

## B1 - heurística de recursos

**Propuesta concreta para revisar:** `score = (opp_prizes - own_prizes) + diff_total_hp / hp_unit + diff_bench_n`. El signo queda positivo cuando el estado favorece al jugador que actúa. `hp_unit` será el promedio de `active_max_hp` positivo observado en los estados de `train`, ponderando para que cada partida tenga el mismo peso. Así, una ventaja en PV se expresa en unidades del HP promedio observado; no elegimos 100 sin evidencia. Este promedio se calcula por escenario y no usa `y` ni los IDs de carta.

Después, una regresión logística de una sola variable aprende en train la pendiente y el intercepto que convierten ese score en probabilidad. No aprende pesos separados para premios, PV y banca. Esta fórmula es una propuesta basada en recursos del juego; hay que revisarla antes de interpretar los resultados como evidencia final.

In [ ]:
# Ejecuta B0 y B1 escenario por escenario para controlar el uso de memoria.
resultados, ajustes = [], []  # Listas para guardar reportes y parámetros finales.
columnas = ['game_id', 'y', 'stage', 'own_prizes', 'opp_prizes', 'own_active_max_hp', 'opp_active_max_hp', 'diff_total_hp', 'diff_bench_n']  # Datos necesarios para métricas y score.
columnas_parquet = [columna for columna in columnas if columna != 'stage']  # stage se calcula desde turn, así que no se lee del Parquet.
for escenario in SCENARIOS:  # Repite el proceso para A, después B y C.
    # ScenarioStream abre un archivo diario cada vez para mantener bajo el uso de RAM.
    stream = ScenarioStream(DATA_DIR, escenario, tuple(columnas_parquet))

    # Calcula la probabilidad base B0 y la escala de HP usando solo train.
    prevalencia, hp_unit = b0_and_hp_unit(stream)

    # Aprende intercepto y pendiente para convertir el score fijo de B1 en probabilidad.
    beta_b1 = fit_b1_mapping(stream, hp_unit, prevalencia)

    # Junta solo val para calcular métricas y guardar sus predicciones B1.
    partes_val = []
    for dia in stream:  # Lee cada archivo diario del escenario.
        val = dia.loc[dia['split'] == 'val', ['game_id', 'y', 'stage']].copy()  # Guarda etiquetas e IDs de las filas de val.
        if len(val):  # Procesa el día solo si contiene estados de val.
            mascara_val = dia['split'] == 'val'  # Encuentra las mismas filas dentro de la tabla diaria completa.
            val['pred_b1'] = b1_probability(dia.loc[mascara_val], hp_unit, beta_b1)  # Calcula probabilidades B1.
            partes_val.append(val)  # Conserva este bloque pequeño de resultados.
    val = pd.concat(partes_val, ignore_index=True)  # Une los bloques diarios de val.
    # Los dos modelos usan los mismos estados reales de val para compararlos justamente.
    resultados += evaluar(val, np.full(len(val), prevalencia), escenario, 'B0')  # B0 repite su probabilidad base.
    resultados += evaluar(val, val['pred_b1'].to_numpy(), escenario, 'B1')  # B1 usa la probabilidad basada en recursos.
    ajustes.append({'escenario': escenario, 'prevalencia_B0_train': prevalencia, 'hp_unit_train': hp_unit,  # Parámetros calculados con train.
                    'intercepto_B1': beta_b1[0], 'pendiente_B1': beta_b1[1],  # Ajustan score → probabilidad.
                    'partidas_val': val.game_id.nunique()})  # Número de partidas evaluadas.
    print(f'Escenario {escenario}: listo')  # Avisa que acabó este escenario.
    del stream, partes_val, val, beta_b1  # Elimina objetos grandes antes del escenario siguiente.
    gc.collect()  # Pide a Python liberar memoria de objetos eliminados.

metricas_baselines = pd.DataFrame(resultados)  # Convierte reportes a tabla.
ajustes_baselines = pd.DataFrame(ajustes)  # Convierte parámetros a tabla.
display(metricas_baselines.round(4))  # Muestra métricas con cuatro decimales.
display(ajustes_baselines.round(4))  # Muestra parámetros con cuatro decimales.
metricas_baselines.to_csv('/content/metricas_B0_B1.csv', index=False)  # Guarda métricas como archivo descargable.
ajustes_baselines.to_csv('/content/ajustes_B0_B1.csv', index=False)  # Guarda parámetros y cantidades evaluadas.
print('Resúmenes guardados en /content/metricas_B0_B1.csv y /content/ajustes_B0_B1.csv')  # Informa dónde se guardaron los CSV.


## Leer los resultados

AUC más alto indica mejor ordenamiento de estados; log loss y Brier más bajos indican mejores probabilidades. `n_partidas` dice cuántas partidas contribuyeron. `pendiente_B1` e `intercepto_B1` ajustan la conversión del score a probabilidad; no son pesos aprendidos para las tres características.

La evaluación aquí es solo en val. No cambies la fórmula mirando test.